# Route B: BMZ BirdNET

CPU is enough. Runtime, Run all.

If `/content/audio/` has no wav, this notebook writes a 120 s synthetic clip and runs BirdNET on it.

## 1. Clone

In [ ]:
REPO = "https://github.com/ST-48-1240162/bioacoustic-embedding-dynamics.git"

%cd /content
!rm -rf bioacoustic-embedding-dynamics
!git clone --depth 1 {REPO}
%cd bioacoustic-embedding-dynamics

## 2. Install

In [ ]:
import sys
!{sys.executable} -m pip install -q soundfile "bioacoustics-model-zoo[birdnet]"
!{sys.executable} -m pip install -q -r docs/colab-requirements.txt
!{sys.executable} -m pip install -q -e .

## 3. Audio

In [ ]:
from pathlib import Path

import numpy as np
import soundfile as sf

AUDIO_DIR = Path("/content/audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
audio_files = sorted(AUDIO_DIR.glob("*.wav")) + sorted(AUDIO_DIR.glob("*.WAV"))
if not audio_files:
    wav = AUDIO_DIR / "sample.wav"
    sr = 48000
    duration = 120.0
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    rng = np.random.default_rng(0)
    y = 0.2 * np.sin(2 * np.pi * 880 * t) + 0.05 * rng.standard_normal(t.size)
    sf.write(wav, y.astype(np.float32), sr)
    audio_files = [wav]
print(len(audio_files), "file(s)")

## 4. Manifest and analysis

In [ ]:
from pathlib import Path

from bioacoustic_embedding_dynamics.adapters import bmz_birdnet_to_manifest

MANIFEST = Path("data/bmz_birdnet.jsonl")
bmz_birdnet_to_manifest(audio_files, MANIFEST, batch_size=8, min_confidence=0.0)
print("lines:", sum(1 for _ in MANIFEST.open()))

In [ ]:
!python -m bioacoustic_embedding_dynamics.cli --manifest {MANIFEST} --out reports/bmz --seed 42

## 5. Summary and figures

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

summary = json.loads(Path("reports/bmz/summary.json").read_text())
print(json.dumps(summary, indent=2))

for name in [
    "pca_species.png", "umap_species.png", "trajectory_pca.png",
    "changepoints.png", "trajectory_changepoints.png", "hmm_regimes.png",
]:
    display(Image(filename=str(Path("reports/bmz") / name)))